# Four Ways to Represent What a Machine Knows
## Semantic Networks, Frames, Propositional Logic and Predicate Logic

### Practical Implementation in Python

**Objective:** Represent the *same* piece of knowledge four different ways, and compare the four representations on:

1. What each one can express
2. What each one can infer
3. How much effort it takes to add new knowledge

> This notebook is designed to be easy for students to understand. Run the cells from top to bottom.

# 1. Learning Objectives

After completing this practical, you should be able to:

- Explain what **knowledge representation** means and why AI needs it.
- Build a **semantic network** from subject-relation-object triples.
- Explain **inheritance** and implement it by following `is_a` links.
- Build a **frame** with slots, defaults and a parent.
- Explain **default reasoning** and how a default is overridden.
- Use **propositional logic** and apply **Modus Ponens**.
- Explain why propositional logic does not scale, and what **predicate logic** adds.
- Use **variables**, **quantifiers** and **substitution** in first-order logic.
- Choose a suitable representation for a given problem.

# 2. Problem Statement

We want a machine to know some very ordinary facts about animals:

```text
   Every bird is an animal.
   Every bird can fly.
   Tweety is a bird.
   Penguins are birds, but a penguin cannot fly.
   Pingu is a penguin.
```

From these we want the machine to work out things nobody told it directly:

> **Can Tweety fly? Can Pingu fly? Is Pingu an animal?**

Nobody stated any of those three facts. The machine has to **derive** them.

This is the whole job of knowledge representation: store facts in a form that
lets a program reason with them.

> The penguin is deliberately awkward. A representation that cannot cope with a
> bird that does not fly is not much use in the real world.

# 3. What is Knowledge Representation?

Knowledge representation is the study of how to write down what a system knows so
that a program can use it.

```text
   Knowledge in a human head        Knowledge a machine can use

   "Birds can fly, and              bird --can--> fly
    Tweety is a bird, so            tweety --is_a--> bird
    Tweety can fly."                => derive: tweety --can--> fly
```

A good representation has to balance four things:

- **Expressiveness** - how much can it say?
- **Inference** - what new facts can be derived from it?
- **Efficiency** - how fast can those facts be found?
- **Clarity** - can a human read it and check it?

No single representation wins on all four, which is exactly why we study several.
We will now build the same animal knowledge four ways.

# 4. Method 1: Semantic Network

A **semantic network** is a graph. Things are **nodes**, and relationships are
**labelled edges** between them.

```text
                    animal
                      ^
                      | is_a
                      |
        can          bird          is_a
   fly <----------    ^   ^   ----------- penguin
                      |    \                  ^
                 is_a |     \ is_a            | is_a
                      |      \                |
                   tweety   (others)        pingu
```

Every edge is stored as a simple three-part statement:

```text
   (subject, relation, object)

   ("bird",   "is_a", "animal")
   ("bird",   "can",  "fly")
   ("tweety", "is_a", "bird")
```

This is the same idea used today by knowledge graphs and the RDF triples behind
the semantic web.

In [ ]:
# A semantic network is just a list of (subject, relation, object) triples.

semantic_network = [
    ("bird",    "is_a", "animal"),
    ("bird",    "can",  "fly"),
    ("bird",    "has",  "feathers"),
    ("penguin", "is_a", "bird"),
    ("penguin", "can",  "swim"),
    ("tweety",  "is_a", "bird"),
    ("pingu",   "is_a", "penguin"),
    ("animal",  "needs", "food"),
]

print("Semantic network:\n")
for subject, relation, obj in semantic_network:
    print(f"  {subject} --{relation}--> {obj}")

print()
print("Number of triples:", len(semantic_network))

## How Inheritance Works

The power of a semantic network comes from following `is_a` links upwards.

To answer *"what can Tweety do?"* we do not look only at Tweety. We climb:

```text
   tweety --is_a--> bird --is_a--> animal

   Collect properties at every level:

   from bird   : can fly, has feathers
   from animal : needs food
```

Tweety inherits everything its ancestors have. This is why a semantic network can
answer questions nobody explicitly stored.

In [ ]:
def find_ancestors(thing, network):
    """Follow is_a links upwards and collect every ancestor."""
    ancestors = []
    current = thing

    while True:
        parent = None
        for subject, relation, obj in network:
            if subject == current and relation == "is_a":
                parent = obj
                break

        if parent is None:
            break

        ancestors.append(parent)
        current = parent

    return ancestors


def ask(thing, relation, network):
    """Find a property of `thing`, inheriting from its ancestors if needed."""
    # Check the thing itself first, then each ancestor in turn
    for level in [thing] + find_ancestors(thing, network):
        for subject, rel, obj in network:
            if subject == level and rel == relation:
                return obj, level
    return None, None

# 5. Run the Semantic Network

In [ ]:
print("===== SEMANTIC NETWORK EXECUTION =====\n")

print("Ancestors of tweety:", find_ancestors("tweety", semantic_network))
print("Ancestors of pingu :", find_ancestors("pingu", semantic_network))
print()

for animal in ["tweety", "pingu"]:
    for relation in ["can", "has", "needs"]:
        answer, source = ask(animal, relation, semantic_network)
        if answer:
            if source == animal:
                print(f"{animal} {relation} {answer}   (stated directly)")
            else:
                print(f"{animal} {relation} {answer}   (inherited from {source})")
    print()

Look at what the machine worked out.

Nobody ever wrote down that Tweety needs food. That was inherited two levels up,
through `tweety -> bird -> animal`.

But now look at Pingu, and at the problem this representation has.

Pingu is reported as **can swim**, and never as *can fly* — which looks like the
right answer. It is not, and the reason matters.

Our `ask` function returns the **first** match it finds. It found
`penguin can swim` and stopped, so the inherited `bird can fly` was silently
dropped. Pingu did not avoid flying because the network knows penguins cannot
fly. Pingu avoided flying because a lookup that returns one value per relation
happened to find something else first.

Two things are genuinely missing here:

- **There is no way to say NO.** The network can state `penguin can swim`. It has
  no notation for *"a penguin cannot fly, even though a bird can"*. A missing
  edge is the only way to express a negative, and a missing edge is
  indistinguishable from knowledge nobody has got round to adding.
- **A relation can hold several values at once.** A real penguin both swims and
  walks. As soon as we want both, returning the first match is clearly wrong, and
  returning all of them brings `fly` straight back.

That is the weakness of a plain semantic network, and it is exactly what frames
fix.

# 6. Method 2: Frames

A **frame** describes one kind of thing as a bundle of named **slots**.

```text
   FRAME: bird                 FRAME: penguin
   +---------------------+     +---------------------+
   | parent   : animal   |     | parent   : bird     |
   | can_fly  : yes      |     | can_fly  : NO       |  <- overrides bird
   | covering : feathers |     | can_swim : yes      |
   | legs     : 2        |     +---------------------+
   +---------------------+
```

Two ideas make frames more powerful than a plain network.

- **Defaults.** A slot value is what is *normally* true. `bird.can_fly = yes` is
  a default, not a law.
- **Overriding.** A more specific frame may replace an inherited value.
  `penguin.can_fly = no` beats `bird.can_fly = yes`.

Looking a value up means starting at the most specific frame and walking up the
`parent` chain until a slot is found. Because we stop at the **first** frame that
has the slot, the most specific answer always wins.

In [ ]:
frames = {
    "animal": {
        "parent": None,
        "needs_food": "yes",
        "alive": "yes",
    },
    "bird": {
        "parent": "animal",
        "can_fly": "yes",          # a DEFAULT, not a law
        "covering": "feathers",
        "legs": 2,
    },
    "penguin": {
        "parent": "bird",
        "can_fly": "no",           # OVERRIDES the bird default
        "can_swim": "yes",
        "habitat": "antarctic",
    },
    "tweety": {
        "parent": "bird",
    },
    "pingu": {
        "parent": "penguin",
    },
}

print("Frames defined:", list(frames.keys()))
print()
for name in ["bird", "penguin"]:
    print(f"FRAME: {name}")
    for slot, value in frames[name].items():
        print(f"   {slot:12s} : {value}")
    print()

In [ ]:
def get_slot(frame_name, slot, frames):
    """Find a slot value, walking up the parent chain. Most specific wins."""
    current = frame_name

    while current is not None:
        frame = frames[current]

        if slot in frame and slot != "parent":
            return frame[slot], current

        current = frame["parent"]

    return None, None

# 7. Run the Frame System

In [ ]:
print("===== FRAME SYSTEM EXECUTION =====\n")

questions = ["can_fly", "covering", "needs_food", "can_swim"]

for animal in ["tweety", "pingu"]:
    print(f"--- {animal} ---")
    for slot in questions:
        value, source = get_slot(animal, slot, frames)
        if value is None:
            print(f"  {slot:12s} : not known")
        elif source == animal:
            print(f"  {slot:12s} : {value}   (stated directly)")
        else:
            print(f"  {slot:12s} : {value}   (inherited from {source})")
    print()

This is the result the semantic network could not produce reliably.

**Tweety can fly.** It has no `can_fly` slot of its own, so the value was
inherited from `bird`, where the default says yes.

**Pingu cannot fly.** The lookup walked `pingu -> penguin` and found `can_fly:
no` at the penguin level. It stopped there and never reached `bird`, so the
default was **overridden** rather than contradicted.

Both animals inherit `needs_food` from `animal`, three levels up.

> This is called **default reasoning**, and it is much closer to how people
> actually think. We believe birds fly, and we abandon that belief for the
> specific birds we know are exceptions — without ever deciding that "birds fly"
> was wrong.

# 8. Method 3: Propositional Logic

Networks and frames store *things*. Logic stores *statements that are true or
false*, and gives us rules for combining them.

A **proposition** is a statement that is either True or False:

```text
   P : "Tweety is a bird"       True
   Q : "Tweety can fly"         True
   R : "Pingu can fly"          False
```

## Logical Connectives

| Symbol | Name | Meaning |
|---|---|---|
| NOT | Negation | opposite of |
| AND | Conjunction | both are true |
| OR | Disjunction | at least one is true |
| IMPLIES | Implication | if the first, then the second |

The one that matters for reasoning is **implication**, written `P -> Q`.

## Modus Ponens

This is the fundamental rule of inference in all of logic:

```text
   If we know:   P -> Q      ("if it is a bird, it can fly")
   And we know:  P           ("it is a bird")
   Then:         Q           ("it can fly")
```

In [ ]:
# Propositions are just True/False values
propositions = {
    "tweety_is_bird": True,
    "tweety_can_fly": None,        # not yet known - we will derive it
    "pingu_is_penguin": True,
}


def IMPLIES(a, b):
    """a -> b is false only when a is True and b is False."""
    return (not a) or b


print("Truth table for P -> Q\n")
print("   P       Q       P -> Q")
print("   " + "-" * 26)
for p in [True, False]:
    for q in [True, False]:
        print("   %-7s %-7s %s" % (p, q, IMPLIES(p, q)))

In [ ]:
def modus_ponens(implication_holds, premise, name_p, name_q):
    """If (P -> Q) and P are both true, conclude Q."""
    print(f"Rule    : {name_p} -> {name_q}   is {implication_holds}")
    print(f"Premise : {name_p}   is {premise}")

    if implication_holds and premise:
        print(f"Conclude: {name_q}   is True")
        return True

    print(f"Conclude: nothing can be derived about {name_q}")
    return None

# 9. Run Propositional Inference

In [ ]:
print("===== PROPOSITIONAL LOGIC EXECUTION =====\n")

# The expert states the rule: being a bird implies being able to fly
bird_implies_fly = True

result = modus_ponens(bird_implies_fly,
                      propositions["tweety_is_bird"],
                      "tweety_is_bird",
                      "tweety_can_fly")

propositions["tweety_can_fly"] = result

print()
print("Knowledge base is now:")
for name, value in propositions.items():
    print(f"  {name:20s} = {value}")

Modus Ponens worked, and it is completely rigorous — but look closely at the
names of those propositions.

`tweety_is_bird` is a single indivisible symbol. Propositional logic cannot see
that it is about *Tweety*, or that it involves *being a bird*. To it, the name is
just a label, and it might as well have been `X47`.

That causes a serious practical problem, which the next section is about.

# 10. Method 4: First-Order Predicate Logic

## Why Propositional Logic Is Not Enough

Suppose we have 1,000 birds. In propositional logic we would need:

```text
   bird_1_is_bird  -> bird_1_can_fly
   bird_2_is_bird  -> bird_2_can_fly
   bird_3_is_bird  -> bird_3_can_fly
   ... one separate rule for every single bird ...
```

One thousand rules, all saying the same thing. And a new bird needs a new rule.

**First-order predicate logic** fixes this by adding three things.

- **Predicates** describe a property of an object: `Bird(tweety)`.
- **Variables** stand for any object: `x`.
- **Quantifiers** say how widely a statement applies.

```text
   FOR ALL x :  Bird(x)  ->  CanFly(x)
```

One rule. It covers every bird that exists now and every bird added later.

## Substitution

To use the rule we **substitute** a real object for the variable:

```text
   Rule    : FOR ALL x : Bird(x) -> CanFly(x)
   Fact    : Bird(tweety)
   Match x = tweety
   Derive  : CanFly(tweety)
```

Matching a variable to an object like this is called **unification**, and it is
the engine behind Prolog and every modern logic programming language.

In [ ]:
# Facts are (predicate, object) pairs
facts = [
    ("Bird", "tweety"),
    ("Bird", "polly"),
    ("Bird", "robin"),
    ("Penguin", "pingu"),
]

# One universal rule covers every bird, however many there are
rule = {"for_all": "x",
        "if": "Bird",
        "then": "CanFly"}

print("Facts:")
for predicate, obj in facts:
    print(f"  {predicate}({obj})")

print()
print(f"Rule: FOR ALL {rule['for_all']} : "
      f"{rule['if']}({rule['for_all']}) -> {rule['then']}({rule['for_all']})")

In [ ]:
def apply_universal_rule(rule, facts):
    """Substitute every matching object into the rule and derive new facts."""
    derived = []

    for predicate, obj in facts:
        if predicate == rule["if"]:
            # Unify: the variable x becomes this object
            print(f"  Match {rule['for_all']} = {obj}")
            print(f"    {rule['if']}({obj}) is a fact")
            print(f"    so derive {rule['then']}({obj})")
            derived.append((rule["then"], obj))

    return derived

# 11. Run Predicate Inference

In [ ]:
print("===== PREDICATE LOGIC EXECUTION =====\n")

print("Applying the single universal rule to every fact:\n")

new_facts = apply_universal_rule(rule, facts)

print()
print("Derived facts:")
for predicate, obj in new_facts:
    print(f"  {predicate}({obj})")

print()
print("One rule produced", len(new_facts), "conclusions.")
print("Adding a new bird needs NO new rule - only the fact Bird(name).")

In [ ]:
# Prove the point: add a new bird and re-run. The rule is untouched.
facts.append(("Bird", "eagle"))

print("Added one new fact: Bird(eagle)\n")

new_facts = apply_universal_rule(rule, facts)

print()
print("Now derived", len(new_facts), "conclusions from the SAME single rule.")

That is the whole argument for predicate logic. We added a bird and the knowledge
base grew by exactly one line. In propositional logic it would have grown by a
fact **and** a rule.

> Note that our rule still says every bird can fly, so it would wrongly conclude
> `CanFly(pingu)` if we recorded Pingu as a bird. Real systems handle this with
> **default logic** or by adding an explicit exception such as
> `FOR ALL x : Penguin(x) -> NOT CanFly(x)`. Logic gives you rigour; it does not
> give you common sense for free.

# 12. Compare What Each Can Express

We asked three questions at the start. Here is how each representation copes.

In [ ]:
questions = [
    "Is Pingu an animal?          (needs multi-level inheritance)",
    "Can Tweety fly?              (needs a default)",
    "Can Pingu fly?               (needs an EXCEPTION to a default)",
    "Do all 1000 birds fly?       (needs one rule, not 1000)",
]

answers = {
    "Semantic Network": ["Yes", "Yes", "No - cannot express NOT",
                         "No - one edge per bird"],
    "Frames": ["Yes", "Yes", "Yes - override wins", "No - one frame per bird"],
    "Propositional Logic": ["No - no inheritance", "Yes - Modus Ponens",
                            "Yes, if stated separately", "No - 1000 rules"],
    "Predicate Logic": ["Yes, with rules", "Yes - substitution",
                        "Needs an explicit exception", "Yes - ONE rule"],
}

for i, q in enumerate(questions):
    print(q)
    for method in answers:
        print("    %-22s %s" % (method + ":", answers[method][i]))
    print()

# 13. Final Comparison Table

In [ ]:
results = {
    "Metric": [
        "Basic Unit",
        "Main Strength",
        "Handles Inheritance?",
        "Handles Exceptions?",
        "Uses Variables?",
        "Can Prove Things?",
        "Effort to Add One Object"
    ],
    "Semantic Network": [
        "Triple (subject, relation, object)",
        "Easy to draw and read",
        "Yes, via is_a links",
        "Not reliably",
        "No",
        "No, only lookup",
        "One triple"
    ],
    "Frames": [
        "Slot inside a frame",
        "Defaults and overriding",
        "Yes, via parent links",
        "Yes",
        "No",
        "No, only lookup",
        "One frame"
    ],
    "Propositional Logic": [
        "Proposition (True/False)",
        "Rigorous proof",
        "No",
        "Only if stated separately",
        "No",
        "Yes, Modus Ponens",
        "One fact and one rule"
    ],
    "Predicate Logic": [
        "Predicate with arguments",
        "One rule covers everything",
        "Yes, with rules",
        "Needs explicit exception rules",
        "Yes",
        "Yes, with substitution",
        "One fact only"
    ]
}

# Display using pandas if available
try:
    import pandas as pd
    comparison = pd.DataFrame(results)
    display(comparison)
except ImportError:
    for i in range(len(results["Metric"])):
        print(results["Metric"][i])
        for key in ["Semantic Network", "Frames", "Propositional Logic",
                    "Predicate Logic"]:
            print("  %-20s %s" % (key + ":", results[key][i]))
        print()

# 14. Knowledge Representation — Conceptual Comparison

| Feature | Semantic Network | Frames | Propositional Logic | Predicate Logic |
|---|---|---|---|---|
| Structure | Graph | Records with slots | Statements | Statements with arguments |
| Reasoning Style | Follow links | Walk parent chain | Modus Ponens | Substitution and proof |
| Expressive Power | Low | Low to medium | Medium | High |
| Default Reasoning | Weak | Strong | None | Needs extra machinery |
| Human Readability | Very high | High | Medium | Lower |
| Computational Cost | Low | Low | Low | Can be very high |
| Modern Descendant | Knowledge graphs, RDF | Object-oriented classes | Boolean satisfiability | Prolog, ontologies, OWL |

# 15. When to Use Each

### Use a Semantic Network when
1. The knowledge is mostly about **how things relate** to each other.
2. Humans need to look at it and understand it quickly.
3. You are building something like a knowledge graph for search.

### Use Frames when
1. Objects have many **properties with typical values**.
2. You need **defaults** that specific cases can override.
3. The domain is naturally hierarchical.

> Notice that a frame is almost exactly a class in object-oriented programming.
> Slots are attributes, `parent` is inheritance, and overriding a slot is
> overriding a method. Frames from 1970s AI became the objects you already use.

### Use Propositional Logic when
1. The problem is genuinely about **true and false combinations**.
2. You need to prove something rigorously.
3. The number of distinct facts is small.

### Use Predicate Logic when
1. A rule must apply to **many objects at once**.
2. New objects are added frequently.
3. You need real proof, not just lookup.

# 16. Important Limitation

Every representation here shares one weakness: **a human has to write the
knowledge down**.

That is the same knowledge bottleneck we met in Practical 3, and it is why
symbolic AI stalled in the 1980s. Nobody could write enough rules fast enough.

There are two further problems worth naming.

- **The frame problem.** When something changes, what stays the same? If Tweety
  flies to a new tree, its colour, its species and its ability to fly are all
  unaffected — but a logical system must be told that, fact by fact.
- **Common sense is enormous.** "Water is wet." "A dropped glass breaks." Millions
  of such facts sit behind ordinary reasoning and almost none of them are written
  down anywhere.

Modern systems increasingly combine both worlds: a **knowledge graph** for facts
that must be exact and auditable, and a **learned model** for the patterns nobody
can express as rules. That combination is called **neuro-symbolic AI**, and it is
an active research area today.

# 17. Student Exercise

Try the following:

### Exercise 1
Add `ostrich` to the frame system. An ostrich is a bird that cannot fly and can
run fast. Check that `get_slot` reports the override correctly.

### Exercise 2
Add `bat` to the semantic network. A bat is a mammal that can fly. What does
`ask("bat", "has", ...)` return, and is that answer sensible?

### Exercise 3
Change `ask` so that it collects **every** matching value instead of returning
the first one. Run it for `pingu` and `can`. Explain what now goes wrong, and why
frames do not have this problem.

### Exercise 4
Write the predicate-logic rule that would correctly stop Pingu from flying, then
describe in one sentence what the inference engine must do to honour it.

### Exercise 5
Compare, for all four methods:
- Can it answer "Is Pingu an animal?"
- Can it handle the penguin exception?
- How many lines does one new bird cost?

### Exercise 6
Represent this sentence in all four notations and say which one you found
clearest:

> "Every student who submits the assignment passes the practical."

In [ ]:
# Student Practice Area
# Add a new animal to any of the four representations here.

# Example - semantic network:
# semantic_network.append(("ostrich", "is_a", "bird"))

# Example - frames:
# frames["ostrich"] = {"parent": "bird", "can_fly": "no", "can_run": "fast"}
# frames["speedy"] = {"parent": "ostrich"}
# print(get_slot("speedy", "can_fly", frames))

# Example - predicate logic:
# facts.append(("Bird", "sparrow"))
# print(apply_universal_rule(rule, facts))

print("Practice area ready!")

# 18. Conclusion

In this practical, we represented the same animal knowledge four different ways
and compared what each one could do.

### Semantic Networks
- Knowledge is a **graph** of triples
- Inheritance works by following **is_a** links
- Very easy to read and draw
- Handles exceptions poorly

### Frames
- Knowledge is a **bundle of slots** with a parent
- Supports **defaults** and **overriding**
- Handled the penguin correctly where the network did not
- Became object-oriented programming

### Propositional Logic
- Knowledge is **true/false statements**
- Reasoning is rigorous, using **Modus Ponens**
- Cannot look inside a statement, so it cannot generalise
- Needs one rule per object

### First-Order Predicate Logic
- Adds **predicates, variables and quantifiers**
- One rule covers every object, now and in the future
- The most expressive of the four
- The most computationally expensive

### Final Decision

There is no single winner, and choosing one is a design decision.

For knowledge that is mostly **relationships between things**, a semantic network
or knowledge graph is clearest. For objects with **typical properties and
exceptions**, frames are the better fit. For knowledge that must be **proved**
and that applies to **many objects at once**, predicate logic is the right tool.

**Key idea:**

> The representation you choose decides what your system can conclude. A fact
> the notation cannot express is a fact the machine can never know.